# 数据分析主 Notebook

本 Notebook 是项目的唯一主分析入口。所有数据库连接、SQL 执行、数据加载、质量检查、清洗、分析、建模、结果验证和导出，都应通过本文件从上到下完整复现。

## 1. 项目背景与分析目标

### 目的
说明项目背景、业务问题和分析目标。

### 输入
用户需求、业务背景、数据源说明。

### 处理逻辑
明确本项目要回答的问题、服务的决策场景和交付物。

### 输出
项目目标、核心问题、交付范围。

### 注意事项
如果目标、口径或数据权限不明确，必须先补充说明。

## 2. 环境、依赖与配置加载

### 目的
加载项目路径、依赖、配置和随机种子。

### 输入
`configs/analysis_config.yaml`、`configs/database.yaml`。

### 处理逻辑
将项目根目录加入 Python 路径，便于调用 `src/` 下的复用模块。

### 输出
项目路径、配置对象和基础参数。

### 注意事项
`configs/database.yaml` 包含真实连接信息，禁止提交。

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "analysis_config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    analysis_config = yaml.safe_load(file)

RANDOM_SEED = analysis_config["project"].get("random_seed", 42)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT

## 3. 数据库连接与 SQL 执行说明

### 目的
通过固定的数据源 profile 读取配置、创建连接，并执行 SQL 文件。

### 输入
`configs/database.yaml`、`sql/` 目录下的 SQL 文件。

### 处理逻辑
使用 `src.db.connection.create_engine_from_profile` 创建连接，使用 `src.db.sql_runner.run_query_with_metadata` 执行只读查询。

### 输出
数据库连接测试结果、SQL 查询结果 DataFrame、执行元数据。

### 注意事项
默认只执行只读查询。涉及写库、删表、更新、建表或覆盖表时，必须先显式确认风险。

In [ ]:
from src.db.connection import create_engine_from_profile, test_connection
from src.db.sql_runner import run_query_with_metadata

DATABASE_CONFIG_PATH = PROJECT_ROOT / "configs" / "database.yaml"
DATABASE_PROFILE = analysis_config["analysis"].get("database_profile")

# 首次运行前，请先复制 configs/database.example.yaml 为 configs/database.yaml，并选择或填写数据源 profile。
# engine = create_engine_from_profile(DATABASE_CONFIG_PATH, profile=DATABASE_PROFILE)
# test_connection(engine)

In [ ]:
# 示例：执行 SQL 文件并查看元数据。
# example_df, example_meta = run_query_with_metadata(
#     engine=engine,
#     sql_file="sql/extract/01_example_query.sql",
# )
# display(example_meta)
# display(example_df.head())

## 4. 数据源与字段理解

### 目的
说明数据表、字段、粒度、时间范围和核心口径。

### 输入
数据库表结构、SQL 查询结果、数据字典。

### 处理逻辑
梳理字段含义、主键、外键、时间字段、统计粒度和业务口径。

### 输出
数据源说明和字段理解记录。

### 注意事项
核心指标字段必须确认口径后再进入分析。

## 5. 数据加载

### 目的
加载 SQL 查询结果或本地数据文件。

### 输入
数据库查询结果、`data/raw/`、`data/external/`。

### 处理逻辑
读取数据后检查行数、列数和基础字段类型。

### 输出
原始 DataFrame 或中间 DataFrame。

### 注意事项
不得手动修改 `data/raw/` 中的原始数据。

## 6. 数据质量检查

### 目的
检查缺失、重复、异常、主键、时间范围和关键指标。

### 输入
已加载的原始数据或 SQL 查询结果。

### 处理逻辑
检查行列数、字段类型、缺失比例、重复记录、异常值、枚举值和 join 后行数变化。

### 输出
`reports/tables/data_quality_summary.csv`。

### 注意事项
任何删除、过滤、填补、去重都必须记录规则和影响行数。

## 7. 数据清洗与转换

### 目的
执行可复现的数据清洗、字段转换和宽表构建。

### 输入
原始数据、数据质量检查结果、清洗规则。

### 处理逻辑
调用 `src/cleaning/`、`src/features/` 中的函数完成清洗和转换。

### 输出
`data/interim/` 或 `data/processed/` 下的中间或最终数据。

### 注意事项
清洗规则必须说明业务含义，禁止静默删除大量数据。

## 8. 探索性数据分析

### 目的
理解分布、趋势、差异、异常点和潜在假设。

### 输入
清洗后的分析数据。

### 处理逻辑
围绕业务问题进行分布、趋势、分组对比和变量关系分析。

### 输出
探索性图表和中间发现。

### 注意事项
图表必须服务于问题，不堆砌无解释图表。

## 9. 业务分析

### 目的
围绕业务问题完成指标拆解、分组对比或专题分析。

### 输入
清洗后的分析数据和业务指标口径。

### 处理逻辑
计算核心指标，进行趋势、分群、漏斗、留存、复购、渠道或收入分析。

### 输出
业务分析结果表、关键图表和核心发现。

### 注意事项
必须说明指标口径、对比基准、样本规模和业务影响。

## 10. 统计分析，如适用

### 目的
使用统计方法验证假设或量化不确定性。

### 输入
分析数据、业务假设、统计方法选择依据。

### 处理逻辑
根据问题选择描述统计、置信区间、假设检验、回归分析或实验分析。

### 输出
统计检验结果、效应量、置信区间和解释。

### 注意事项
必须区分统计显著性和业务显著性。

## 11. 机器学习建模，如适用

### 目的
在确有预测、分类、聚类或排序需求时完成建模和评估。

### 输入
建模样本、标签、特征、切分方式和评估指标。

### 处理逻辑
调用 `src/modeling/` 中的函数完成特征处理、模型训练、评估和解释。

### 输出
模型评估结果、特征解释、适用边界。

### 注意事项
禁止数据泄漏，禁止为了建模而建模。

## 12. 结果验证与稳健性检查

### 目的
检查结论是否稳定，是否受异常值、时间窗口、样本选择或口径变化影响。

### 输入
核心结论、关键指标、模型或统计结果。

### 处理逻辑
通过改变时间窗口、分组、样本、口径或模型方法验证结果稳定性。

### 输出
稳健性检查结果。

### 注意事项
如果结论高度敏感，必须在最终报告中说明。

## 13. 图表、表格和报告素材导出

### 目的
将关键图表、结果表和报告素材保存到 `reports/`。

### 输入
关键分析结果、图表对象、结果 DataFrame。

### 处理逻辑
统一导出到 `reports/figures/`、`reports/tables/`、`reports/final/`。

### 输出
可交付的图表、表格和报告素材。

### 注意事项
最终结论不能只存在于 Notebook 输出缓存中。

## 14. 最终结论、业务建议与局限性

### 事实
列出数据直接支持的发现。

### 推断
说明基于事实得到的解释。

### 建议
提出可执行业务建议。

### 局限性
说明数据、方法和结论边界。

## 15. 项目完成检查清单

- [ ] 分析目标已明确。
- [ ] 数据源、时间范围和数据粒度已说明。
- [ ] 核心指标口径已说明。
- [ ] SQL 文件已保存到 `sql/` 并由 Notebook 调用。
- [ ] 数据质量检查已完成。
- [ ] 清洗规则和影响行数已记录。
- [ ] 关键图表已导出到 `reports/figures/`。
- [ ] 关键结果表已导出到 `reports/tables/`。
- [ ] 最终报告或结论材料已保存到 `reports/final/`。
- [ ] 结论区分事实、推断、建议和局限性。
- [ ] 未泄露数据库凭据或敏感明细数据。